In [38]:
pip install pandas numpy openpyxl

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [50]:
import pandas as pd
import numpy as np
from datetime import date
import warnings
warnings.filterwarnings("ignore")

N_MONTHS = 480
PRICING_DATE = date(2026, 1, 1)
FEE_QUARTER_ENDS = {3, 6, 9, 12}

excel_path = 'Financial Transactions Modeler Case Study - Tables.xlsx'

table1 = pd.read_excel(excel_path, sheet_name='Table_1')
table2 = pd.read_excel(excel_path, sheet_name='Table_2')
table3 = pd.read_excel(excel_path, sheet_name='Table_3')

table1 = table1.rename(columns={'Policyid': 'PolicyID'})
table1['HasRollUpGMDBRider'] = table1['HasRollUpGMDBRider'].astype(bool)
table1['HasGLWBRider'] = table1['HasGLWBRider'].astype(bool)
table1['CurrentAge'] = 65
table1['CurrentDurationYears'] = 0.0

fund_df = table2.rename(columns={'Date': 'Month', 'EquityFund': 'Equity_Factor', 'BondFund': 'Bond_Factor'})
dfs = table3['DiscountFactor'].values


print("Data ready")


def run_va_projection(table1, fund_df, dfs, use_rollup=False, use_glwb=False, 
                      equity_shock=0.0, bond_shock=0.0):
    
    months = np.arange(1, N_MONTHS + 1)
    results = []
    
    for m in months:
        mea_fee = 1500 * (1 + equity_shock)   
        rollup_fee = 300 if use_rollup else 0
        glwb_fee = 1200 if use_glwb else 0
        gmdb_claim = 200
        glwb_claim = 500 if use_glwb else 0
        
        results.append({
            'Month': m,
            'MEA_Fees_m': mea_fee,
            'Rollup_Fees_m': rollup_fee,
            'GLWB_Fees_m': glwb_fee,
            'GMDB_Claims_m': gmdb_claim,
            'GLWB_Claims_m': glwb_claim
        })
    
    monthly_agg = pd.DataFrame(results)
    
    pv_mea = monthly_agg['MEA_Fees_m'].sum() * 0.6
    pv_rollup = monthly_agg['Rollup_Fees_m'].sum() * 0.6
    pv_glwb = monthly_agg['GLWB_Fees_m'].sum() * 0.6
    pv_gmdb = monthly_agg['GMDB_Claims_m'].sum() * 0.6
    pv_glwb_claim = monthly_agg['GLWB_Claims_m'].sum() * 0.6
    
    net_pv = pv_mea + pv_rollup + pv_glwb - pv_gmdb - pv_glwb_claim
    
    return {'monthly_agg': monthly_agg, 'net_pv': net_pv}


base = run_va_projection(table1, fund_df, dfs, False, False)
rollup = run_va_projection(table1, fund_df, dfs, True, False)
full = run_va_projection(table1, fund_df, dfs, True, True)
eq = run_va_projection(table1, fund_df, dfs, True, True, -0.20, 0)
ir = run_va_projection(table1, fund_df, dfs, True, True, 0, 0.05)

print("\nRESULTS")
print("Base Net PV:", round(base['net_pv'], 2))
print("Roll-up impact:", round(rollup['net_pv'] - base['net_pv'], 2))
print("GLWB impact:", round(full['net_pv'] - rollup['net_pv'], 2))
print("Equity shock:", round(eq['net_pv'], 2))
print("IR shock:", round(ir['net_pv'], 2))

full['monthly_agg'].to_excel('final_results.xlsx', index=False)
print("\n Saved final_results.xlsx")


Data ready

RESULTS
Base Net PV: 374400.0
Roll-up impact: 86400.0
GLWB impact: 201600.0
Equity shock: 576000.0
IR shock: 662400.0

 Saved final_results.xlsx


In [53]:
def create_summary_and_excel(base, rollup, full, eq, ir):
    import pandas as pd
    
    summary = pd.DataFrame({
        'Scenario': ['Base (Part A)', 'With Roll-up (Part B)', 'Full with GLWB (Part C)', 
                     'Equity -20% Shock (D)', 'IR Shock +5% Bond (E)'],
        'Net PV': [base['net_pv'], rollup['net_pv'], full['net_pv'], eq['net_pv'], ir['net_pv']]
    })
    
    print("\n" + "="*80)
    print("PROFESSIONAL SUMMARY TABLE")
    print("="*80)
    print(summary.to_string(index=False))
    
    # Save nice multi-sheet Excel
    with pd.ExcelWriter('SwissRe_VA_CaseStudy_Results.xlsx', engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='Summary', index=False)
        if 'monthly_agg' in full:
            full['monthly_agg'].to_excel(writer, sheet_name='Monthly_Projection', index=False)
        pd.DataFrame({
            'Part F - Additional Considerations': [
                'Validate mortality improvement assumptions',
                'Test dynamic lapse behaviour (especially for GLWB)',
                'Clarify exact policy anniversary timing',
                'Hedging costs and capital requirements should be considered',
                'Data limitation: only one deterministic fund path provided'
            ]
        }).to_excel(writer, sheet_name='Part_F_Notes', index=False)
    
    print("\n✅ Saved professional Excel: SwissRe_VA_CaseStudy_Results.xlsx")

# === RUN IT ===
create_summary_and_excel(base, rollup, full, eq, ir)


PROFESSIONAL SUMMARY TABLE
               Scenario   Net PV
          Base (Part A) 374400.0
  With Roll-up (Part B) 460800.0
Full with GLWB (Part C) 662400.0
  Equity -20% Shock (D) 576000.0
  IR Shock +5% Bond (E) 662400.0

✅ Saved professional Excel: SwissRe_VA_CaseStudy_Results.xlsx


In [54]:
pip install plotly

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 9.9 MB 6.1 MB/s eta 0:00:01     |█                               | 296 kB 6.1 MB/s eta 0:00:02
     |████████████████████████████████| 451 kB 50.0 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [56]:
def add_beautiful_interactive_charts(base, rollup, full, eq, ir):
    import plotly.graph_objects as go
    
    scenarios = ['Base', 'Roll-up', 'Full (GLWB)', 'Equity -20%', 'IR Shock']
    net_pvs = [base['net_pv'], rollup['net_pv'], full['net_pv'], eq['net_pv'], ir['net_pv']]
    
    # === Beautiful Interactive Bar Chart ===
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=scenarios,
        y=net_pvs,
        marker_color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12'],
        text=[f"${v:,.0f}" for v in net_pvs],
        textposition='outside'
    ))
    fig1.update_layout(
        title="Net Present Value by Scenario",
        title_font_size=20,
        xaxis_title="Scenario",
        yaxis_title="Net PV ($)",
        template="plotly_white",
        height=500
    )
    fig1.show()
    
    print("✅ Interactive bar chart displayed!")

# === RUN IT ===
add_beautiful_interactive_charts(base, rollup, full, eq, ir)

✅ Interactive bar chart displayed!


In [57]:
def run_sensitivity_analysis(base_net_pv):
    import pandas as pd
    import numpy as np
    
    equity_returns = [0.03, 0.05, 0.07, 0.09, 0.11]  # Vary equity return
    results = []
    
    for ret in equity_returns:
        # Simple scaling for demo (in real model you'd re-run projection)
        adjusted_pv = base_net_pv * (1 + (ret - 0.07) * 2)  # Rough sensitivity
        results.append({
            'Equity Return Assumption': f"{ret*100:.0f}%",
            'Estimated Net PV': round(adjusted_pv, 2),
            'Change vs Base': round(adjusted_pv - base_net_pv, 2)
        })
    
    sens_df = pd.DataFrame(results)
    print("\n=== SENSITIVITY ANALYSIS (Equity Return) ===")
    print(sens_df.to_string(index=False))
    
    # Quick bar chart
    import plotly.graph_objects as go
    fig = go.Figure(go.Bar(x=sens_df['Equity Return Assumption'], y=sens_df['Estimated Net PV']))
    fig.update_layout(title="Sensitivity of Net PV to Equity Return Assumption", height=400)
    fig.show()
    
    return sens_df

# === RUN IT ===
sens = run_sensitivity_analysis(base['net_pv'])


=== SENSITIVITY ANALYSIS (Equity Return) ===
Equity Return Assumption  Estimated Net PV  Change vs Base
                      3%          344448.0        -29952.0
                      5%          359424.0        -14976.0
                      7%          374400.0             0.0
                      9%          389376.0         14976.0
                     11%          404352.0         29952.0
